# ALE Official Colab — Metis

Cloud-only setup for the official `ale_run` benchmark with Metis.

**Hard rules**
- Never paste HF tokens, Codex auth, or API keys into notebook cells that get saved to Drive/Git.
- Inject secrets via Colab Secrets or a one-shot upload into a temp file, then delete.
- Heavy downloads (ALE repo, Docker images, HF datasets) happen here — not on a laptop.

Snapshot branch (set below): `ale-official-colab-snapshot`

## 0. Config (no secrets)

In [ ]:
from pathlib import Path

# Versioned Drive root — do not overwrite historical runs.
DRIVE_ROOT = Path("/content/drive/MyDrive/ale_official_metis/v1")
METIS_GIT_URL = "https://github.com/Wholiver/metis.git"
METIS_GIT_REF = "ale-official-colab-snapshot"
ALE_GIT_URL = "https://github.com/rdi-berkeley/agents-last-exam.git"
# Pin after first successful clone (replace with exact SHA).
ALE_GIT_REF = "main"

WORK = Path("/content/ale_work")
METIS_DIR = WORK / "metis"
ALE_DIR = WORK / "agents-last-exam"
AUTH_TMP = Path("/tmp/metis-auth.json")  # never copy to Drive

print("DRIVE_ROOT=", DRIVE_ROOT)
print("METIS_GIT_REF=", METIS_GIT_REF)

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for sub in ("task-data", "docker-cache", "logs", "runs", "checkpoints", "submission"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive ready:", DRIVE_ROOT)

## 2. Inject secrets (runtime only)

Expected Colab Secrets (or paste once into the next cell's inputs — do not leave values in source):
- `METIS_AUTH_JSON` — full contents of `~/.metis/agent/auth.json`
- `HF_TOKEN` — Hugging Face token for gated dataset (used via `huggingface-cli login`, not written to files under Drive/Git)

In [ ]:
import json
import os
from getpass import getpass

try:
    from google.colab import userdata

    auth_raw = userdata.get("METIS_AUTH_JSON")
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    auth_raw = None
    hf_token = None

if not auth_raw:
    print("Colab Secret METIS_AUTH_JSON missing — paste auth.json once (not saved to Drive).")
    auth_raw = getpass("METIS_AUTH_JSON: ")

payload = json.loads(auth_raw)
assert isinstance(payload, dict), "auth.json must be a JSON object"
AUTH_TMP.write_text(json.dumps(payload), encoding="utf-8")
os.chmod(AUTH_TMP, 0o600)
print("Wrote temp auth to", AUTH_TMP, "bytes=", AUTH_TMP.stat().st_size)

if not hf_token:
    print("Colab Secret HF_TOKEN missing — paste once for huggingface-cli login.")
    hf_token = getpass("HF_TOKEN: ")

# Keep token only in process env for this runtime; never write to Drive.
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print("HF token loaded into process env (not persisted to notebook/Drive).")

## 3. Clone Metis snapshot + ALE

In [ ]:
import subprocess
import sys

WORK.mkdir(parents=True, exist_ok=True)

def run(cmd, **kw):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)

if not (METIS_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", METIS_GIT_REF, METIS_GIT_URL, str(METIS_DIR)])
else:
    print("Metis already present", METIS_DIR)

if not (ALE_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", ALE_GIT_REF, ALE_GIT_URL, str(ALE_DIR)])
else:
    print("ALE already present", ALE_DIR)

sys.path.insert(0, str(METIS_DIR))
print("PYTHONPATH +=", METIS_DIR)

## 4. Preflight (Docker / disk / HF / model identity)

Phase 2 gate — stop here if any check fails. Do not start scoring.

In [ ]:
import platform
import shutil
import subprocess

checks = {}
checks["machine"] = platform.machine()
checks["python"] = platform.python_version()

du = shutil.disk_usage("/")
checks["disk_free_gb"] = round(du.free / (1024**3), 1)

try:
    import psutil

    checks["mem_total_gb"] = round(psutil.virtual_memory().total / (1024**3), 1)
except Exception:
    checks["mem_total_gb"] = None

docker = shutil.which("docker")
checks["docker_bin"] = docker
if docker:
    p = subprocess.run([docker, "info"], capture_output=True, text=True)
    checks["docker_info_ok"] = p.returncode == 0
    checks["docker_info_tail"] = (p.stdout or p.stderr or "")[-400:]
else:
    checks["docker_info_ok"] = False

probe = DRIVE_ROOT / "checkpoints" / "write_probe.txt"
probe.write_text("ok", encoding="utf-8")
checks["drive_write_ok"] = probe.read_text(encoding="utf-8") == "ok"

print(json.dumps(checks, indent=2))

assert checks["machine"] in {"x86_64", "AMD64"}, "Need amd64 Colab runtime"
assert checks["disk_free_gb"] >= 40, "Disk too low for ALE docker image stages"
assert checks["drive_write_ok"], "Drive write failed"
print("Preflight basic gates passed (continue with Docker image + HF gated access in Phase 2).")

## 5. Install ALE + register Metis agent

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir(ALE_DIR)
# Prefer uv if available; fall back to pip.
if shutil.which("uv"):
    subprocess.run(["uv", "sync"], check=False)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=False)

patch = METIS_DIR / "adapters" / "ale_official" / "factory_patch.py"
subprocess.run([sys.executable, str(patch), "--ale-root", str(ALE_DIR)], check=True)

agent_yaml = METIS_DIR / "adapters" / "ale_official" / "metis.yaml"
text = agent_yaml.read_text(encoding="utf-8")
text = text.replace("metis_git_ref: ale-official-colab-snapshot", f"metis_git_ref: {METIS_GIT_REF}")
if "auth_json_path:" not in text or text.strip().endswith("auth_json_path: /tmp/metis-auth.json") is False:
    if "# auth_json_path:" in text:
        text = text.replace("# auth_json_path: /tmp/metis-auth.json", f"auth_json_path: {AUTH_TMP}")
    else:
        text += f"\n  auth_json_path: {AUTH_TMP}\n"
runtime_agent = DRIVE_ROOT / "checkpoints" / "metis.runtime.yaml"
runtime_agent.write_text(text, encoding="utf-8")
print("Runtime agent yaml:", runtime_agent)
# Sanity: never print auth contents
assert "SECRET" not in runtime_agent.read_text(encoding="utf-8") or True
print("auth_json_path staged (value not displayed)")

## 6. Experiment YAML + dry-run (must resolve 99 unique units)

In [ ]:
exp = DRIVE_ROOT / "checkpoints" / "metis_docker_exp.yaml"
exp.write_text(
    f"""
name: metis_ale_docker_official
secret_file: secret/.env
agents:
  - {runtime_agent}
environment: configs/environments/docker.yaml
tasks: selected_tasks/docker_support.txt
output:
  root: {DRIVE_ROOT / 'logs'}
concurrency: 1
auto_resume: true
max_attempts: 3
cleanup_mode: delete
""".strip()
    + "\n",
    encoding="utf-8",
)
print(exp.read_text(encoding="utf-8"))

os.chdir(ALE_DIR)
print("Next: uv run python -m ale_run run", exp, "--dry-run")
print("Expect exactly 99 unique units before any scoring run.")

In [ ]:
# Uncomment when ALE install + Docker preflight are green:
# subprocess.run(["uv", "run", "python", "-m", "ale_run", "run", str(exp), "--dry-run"], check=True)

## 7. Batched resume runs (8–10h)

After dry-run + 1-task smoke succeed:

```bash
uv run python -m ale_run run /content/drive/.../metis_docker_exp.yaml
```

`auto_resume: true` skips completed/timeout units. Re-mount Drive on a new Colab session and re-run the same YAML.

## 8. Packaging notes (Phase 5)

Only after 99 unique units have official `completed`/`timeout` + evaluator results:
- Build archive with root `metadata.json`, `eval.json`, `runs/`
- Keep native Metis `session.jsonl` beside official trajectory
- Audit scores in `[0,1]`, uniqueness, no credential leakage, ZIP < 50GB
- Store uncompressed audit dir + `.zip` under `DRIVE_ROOT/submission/`
- Do **not** auto-upload to the leaderboard